In [52]:
%%capture
!pip install fuzzywuzzy
!pip install python-Levenshtein
!pip install recordlinkage
!pip install unidecode
!pip install joypy

In [53]:
# Importa as bibliotecas Google Sheets
import gspread
import pandas as pd
import recordlinkage
import re
import unidecode
from fuzzywuzzy import fuzz
from fuzzywuzzy import process
import joypy
from google.colab import auth
auth.authenticate_user()
from google.auth import default
from gspread_dataframe import set_with_dataframe
creds, _ = default()
gc = gspread.authorize(creds)

In [54]:
# VARIÁVEIS

# Dados de onde está localizada a planilha com as Notas do EUF (ORIGINAL: '1G2r-O-adeP_YjRzlR1U0IskSSWv0OHy5')
id_pasta_1 = '16KwgyZ2KySE1V93mQsdy5wRATwvZuVy3'  # id da pasta onde está localizada a Planilha no seu Google drive
nome_planilha_1 = 'Notas EUF - 2006 em diante' # Nome da Planilha dentro do seu Google drive


# Dados de onde está localizada a planilha com as inscrições do CPGpar (ORIGINAL: '1G2r-O-adeP_YjRzlR1U0IskSSWv0OHy5')
id_pasta_2 = '16KwgyZ2KySE1V93mQsdy5wRATwvZuVy3'  # id da pasta onde está localizada a Planilha no seu Google drive
nome_planilha_2 = 'Inscrições_CPGpar' # Nome do Planilha dentro do seu Google drive


# Nome da edição atual que será usado para salvar os arquivos dos aprovados e reprovados
edicao = "1_2025"  # Nome do Planilha dentro do seu Google drive

In [55]:
# Função para unir nomes semelhantes, com base em uma %, entre o dfA (CPGpar) com o dfB(Notas do EUF)
# Retorna dfA combinado com as colunas escolhidas (colunas_B) de dfB.
def merge_RL(dfA,dfB,nome_B,colunas_B,nivel):
  # candidatos
  indexer = recordlinkage.Index()
  indexer.full()
  canditados = indexer.index(dfA, dfB)
  # Comparação
  compare_cl = recordlinkage.Compare()
  compare_cl.string('Nome_', 'Nome', method='levenshtein', threshold=nivel, label= 'conta' )
  features = compare_cl.compute(canditados, dfA, dfB)
  matches = features[features.sum(axis=1) >= 1]
  matches = matches.reset_index()
  dfA = dfA.reset_index()

  merge1=dfA.merge(matches,
                left_on='index',
                right_on='level_0',
                how='left')

  matches=merge1.merge(dfB[lista_colunas],
                left_on = 'level_1',
                right_index=True,
                suffixes=(None, None),
                how='left'
  )
  matches = matches.drop(columns = ['index','level_0','level_1','conta'])
  return matches

# Funçao que recebe um dataframe e sua coluna com as notas
# Retorna o mesmo dataframe com uma nova coluna (_Norm) com a nota normalizada
def notaNormalizada(df,coluna):
  df = eval(df)
  notas = pd.to_numeric(df[coluna])
  notas = df[coluna].astype(float)
  df[coluna+'_Norm'] = notas*5/notas.mean()
  return df

# Funçao que recebe uma  string e um caracter (subtring)
# Retorna a mesma string sem o caracter
def limpa_nota(string,subtring):
  # transforma
  string = str(string).replace(subtring,"")
  return string

# Funçao que recebe a nota normalizada
# Retorna "Aprovado"se >4 ou "Sem nota suficiente"
def verificaNotaNorm(nota_normalizada):
  if nota_normalizada >=4.00:
    return "Aprovado"
  else:
    return "Sem nota suficiente"

# Funçao que recebe a nota normalizada
# Retorna "Aprovado"se >6 ou "Sem nota suficiente"
def verificaNotaNormDD(nota_normalizada):
  if nota_normalizada >=6.00:
    return "Aprovado"
  else:
    return "Sem nota suficiente"

# Funçao que recebe o texto
# Retorna o texto sem ","
def removeVirgulaGS(rows):
  for i, x in enumerate(rows):
      for j, a in enumerate(x):
          if ',' in a:
              rows[i][j] = a.replace(',', '.')
  return rows

# Funçao que recebe um dataframe
# Retorna o df
def etiquetaDFinteiro(df,etiqueta):
  etiqueta=str("_"+etiqueta+"")
  df.columns=[x + etiqueta for x in list(df.columns)]
  return df


# Funçao que recebe o dataframe e 2 colunas
# Retorna uma lista com os valores do dataframe
def subtringLista(lista,substring1, substring2=None):
  lista_filtrada = []
  for string in lista:
    if substring2 != None:
      if substring1 in string and substring2 not in string:
        lista_filtrada.append(string)
      else:
        pass
    else:
      if substring1 in string:
        lista_filtrada.append(string)
      else:
        pass
  return lista_filtrada


# Funçao que recebe um texto
# Retorna o texto sem aspas
def extraiNomePlanilha(s):
  padrao = "\'(.*?)\'"
  return re.search(padrao,s).group(1)


# Funçao que recebe um texto
# Retorna o texto com a primeira letra de cada palavra maiúscula
def capitalizarNome(string):
  return ' '.join([s.capitalize() for s in string.split()])


# Funçao que recebe um dataframe e colunas especificas
# retorna o mesmo dataframe com o sufixo "_" no nome das colunas específicas
def etiqueta_df(df, colunas, etiqueta):
  etiqueta=str("_" + etiqueta)
  colunas_etiq = [x + etiqueta for x in colunas]
  dicionario = dict(zip(colunas, colunas_etiq))
  df=df.rename(columns=dicionario)
  return df


# Funçao que recebe um dataframe
# Retorna o arquivo em google sheets
def DFParaGsheet(nome_arquivo, nome_planilha, df):
  n_col, n_linha = df.shape
  lista_arquivos = [d['name'] for d in gc.list_spreadsheet_files()]
  if nome_arquivo not in lista_arquivos:
    sh=gc.create(nome_arquivo,folder_id=id_folder)
  else:
    sh = gc.open(nome_arquivo)
  planilha = sh.add_worksheet(title=nome_planilha, rows=n_linha, cols=n_col)
  planilha.update([df.columns.values.tolist()] + df.values.tolist())


# Funçao que recebe um texto
# Retorna Mestrado, Doutorado ou Dou.Direto
def corrigeCurso(string):
  if string =='inscricaomd':
    return "Mestrado"
  elif string == 'incricaodr':
    return "Doutorado"
  else:
    return "Dou.Direto"

# Funçao que recebe um texto
# Retorna Sim ou Não
def corrigeConcorreBolsa(string):
  if string == 'Desejo concorrer à Bolsa':
    return "Sim"
  else:
    return "Não"


# Funçao que recebe um texto
# Retorna o texto sem virgulas
def removeVirgulaGS(rows):
  for i, x in enumerate(rows):
      for j, a in enumerate(x):
          if ',' in a:
              rows[i][j] = a.replace(',', '.')
  return rows


# Funçao que recebe um texto
# Retorna o texto sem espaços adicionais
def removeMultEspacos(string: str) -> str:
    string_arrumada = re.sub(' +', ' ', string)
    return string_arrumada


# Funçao que recebe um texto
# Retorna o texto sem acentos
def removeAcentos(string: str) -> str:
    string_arrumada = unidecode.unidecode(string)
    return string_arrumada


# Funçao que recebe um texto
# Retorna o texto sem acentos
# Retorna o texto sem acentos e sem espaços a direita e a esquerda
def processa_texto(df: pd.DataFrame, coluna: list) -> pd.DataFrame:
    coluna=str(coluna)
    df[coluna] = df[coluna].apply(str.rstrip) # remove espaços a direita em strings
    df[coluna] = df[coluna].apply(str.lower) # strings em letas minúsculas
    df[coluna] = df[coluna].apply(str.lstrip) # remove espaços a esqueda em strings
    df[coluna] = df[coluna].apply(removeAcentos) # remove acentuação
    df[coluna] = df[coluna].apply(removeMultEspaços) # remove espaços multiplos: maior ou igual a 2
    return df

# Funçao que recebe um texto
# Retorna o texto tudo em minúsculo, sem acentos, sem espaços a direita e a esquerda e sem multiplo espaços
def processaString(string) -> str:
    string = string.lower()
    string = removeMultEspacos(string)# remove espaços multiplos: maior ou igual a 2
    string = string.strip() # remove espaços a direita  e esqueda em strings
    string = removeAcentos(string)
    return string

# Importação dos dados

In [56]:
# Abre planilha do Google Sheet
sh1 = gc.open(nome_planilha_1, folder_id= id_pasta_1)


# Lista de listas que armazena todos os dados de cada aba em dataframes separados
dic={}

lista_p = [extraiNomePlanilha(str(i)) for i in sh1.worksheets()] # Extrai o nome de "todas as abas" dentro da planilha e salva na lista_p
for p in lista_p:
  planilha = sh1.worksheet(p)
  rows = planilha.get_all_values() # pega os valores preenchidas com algum valor das linhas da planilha
  rows = removeVirgulaGS(rows)
  dataframe=pd.DataFrame.from_records(rows[1:],columns=rows[0])
  dic[p] = dataframe  # armazena cada aba em uma lista

dfs_euf = list(dic.keys()) #  dfs_euf é o nome com todos os dataframes criados no dicionario "dic"
try:
  dfs_euf.remove('2021_2') #  remove edição cancelada do EUF caso ela esteja erroneamente na planilha de notas.
except:
  pass

In [57]:
# não precisa execultar
for key in dfs_euf[:]:  #Verifica e lista todos as edições de EUF
  print(key)

2025_2
2025_1
2024_2
2024_1
2023_2
2023_1
2022_2
2022_1
2021_3
2021_1
2020_3
2020_2
2020_1
2019_2
2019_1
2018_2 (2019-1)
2018_1 (2018-2)
2017_2 (2018-1)
2017_1 (2017-2)
2016_2 (2017-1)
2016_1 (2016-2)
2005_2
2005_1
2006_1
2006_2


In [58]:
# não precisa execultar
dic['2025_1'] # apenas para visualizar os dados de alguma aba

,EUF CODE,Nome,Classificação,Nota,Eletromagnetismo Nota,Eletromagnetismo Classificação,Física Estatística Nota,Física Estatística Classificação,Física Moderna Nota,Física Moderna Classificação,Mecânica Clássica Nota,Mecânica Clássica Classificação,Mecânica Quântica Nota,Mecânica Quântica Classificação,Termodinâmica Nota,Termodinâmica Classificação,Percentil
0,12025EUF0501,Abner Vinicius Oles de Mello,1,2.5,0.5,1,0.25,1,0.5,1,0.25,1,1,3,0,1,24.30%
1,12025EUF1007,Abraão de Barros Marreira Filho,4,6.5,1.5,4,0.25,1,1.5,4,1.25,4,1.25,3,0.75,4,93.38%
2,12025EUF0887,Adèle MEGUEDONG YOTA,1,2,0.5,1,0,1,0.75,2,0.5,2,0.25,1,0,1,11.51%
3,12025EUF0301,ADENILSON SOUZA CHAVES MATOS,1,2,0.25,1,0.25,1,0.75,2,0,1,0.75,2,0,1,11.51%
4,12025EUF0820,Adriane Martins Alves,2,3,0.25,1,0.5,2,0.5,1,0.5,2,1,3,0.25,2,38.71%
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1098,12025EUF1103,Yuri da Silva Avila,2,3.25,0.75,2,0,1,0.75,2,0.75,3,0.75,2,0.25,2,46.24%
1099,12025EUF0818,Yuri da Silva Borges,1,1.5,0.5,1,0,1,0.75,2,0,1,0.25,1,0,1,4.53%
1100,12025EUF1514,Yuri Rodrigues Batista,1,1.25,0.5,1,0,1,0.5,1,0.25,1,0,1,0,1,1.99%
1101,12025EUF1476,Yves Ferreira Vicente,1,2.75,0,1,0.25,1,1,3,0.5,2,0.75,2,0.25,2,24.91%


In [59]:
# Abre planilha do Google Sheet
sh2 = gc.open(nome_planilha_2, folder_id= id_pasta_2) # abre o arquivo Google Sheet
planilha2 = sh2.worksheet("Inscritos")

# ATENÇÃO!!! O método .get_all_values() pode falhar e exibir: exceeds grid limits
# renomeie a planilha adicionando underscore "_" em alguma parte no nome. Funciona, não se sabe porque.

rows = planilha2.get_all_values() # pega os valores preenchidas com algum valor das linhas da planilha
CPGPAR = pd.DataFrame.from_records(rows[1:], columns=rows[0]) # transforma em data frame do pandas


# Cabeçalho para trocar o nome de algumas colunas (dicionário)
novos_cabecalhos = {'curso no qual o candidato está se inscrevendo' : 'Curso',
                   'Instituição de onde vem' : 'Inst. origem',
                   'Selecione o EUF cuja nota você pretende utilizar para ingresso' : 'EUF indicado',
                   'Nota antiga do Exame EUF' : 'EUF antigo',
                   'Número de Inscrição' : 'Nº de inscrição',
                   'Declaro que' : "Concorre à bolsa"
}

# Aplica o cabeçalho
CPGPAR = CPGPAR.rename(columns=novos_cabecalhos)

CPGPAR['Curso'] = CPGPAR['Curso'].apply(corrigeCurso)
CPGPAR['Concorre à bolsa'] = CPGPAR['Concorre à bolsa'].apply(corrigeConcorreBolsa)
CPGPAR['Nome'] = CPGPAR['Nome'].apply(processaString)

In [60]:
# não precisa rodar
CPGPAR # apenas para visualizar os dados de alguma aba

,Nome,Curso,Nº de inscrição,Inst. origem,Email,Concorre à bolsa,EUF indicado,EUF antigo,Orientador Informado,Tem carta de aceite?,...,Observações,Data de Envio,Sexo Biológico,Identidade de Gênero,Orientação Sexual,Pessoa com Deficiência (PcD*)?,Cor ou raça*,Nível educacional da mãe ou genitor primário,Onde estudou a mãe (ou genitor primário) no ensino médio?,Onde estudou no ensino médio?
0,alejandro lopez guilhermino,Mestrado,14058,,aleh.lopezg@gmail.com,Sim,,,Ana Julia Silveira Mizher,Sim,...,Média de notas calculada faltando o último sem...,07/11/2025 - 13:31,Masculino,Cisgênero (se identifica com o gênero associad...,Bissexual,Não,Branca,Superior completo,,Parte ou todo em escolas técnicas Federais ou ...
1,alex sander do carmo souza,Mestrado,13824,,alexsander6001@gmail.com,Sim,,,Elcio Abdalla,Nao,...,"informou como orientador Elcio Abdalla, mas nã...",31/10/2025 - 20:45,Masculino,Cisgênero (se identifica com o gênero associad...,Heterossexual,Não,Parda,Médio incompleto,,Parte ou todo em escolas técnicas Federais ou ...
2,alexia oliveira silva,Doutorado,13847,,alexia.o@usp.br,Sim,,,Luiz Gustavo Pimenta Martins,Sim,...,,03/11/2025 - 08:43,Feminino,Cisgênero (se identifica com o gênero associad...,Heterossexual,Não,Preta,Superior completo,,Todo em escola pública (não técnica)
3,alice affonso sassmannshausen,Mestrado,14023,,alicesass1009@hotmail.com,Sim,,,,,...,,07/11/2025 - 01:12,,,,,,,,
4,aline paulo da costa,Mestrado,13859,,aline_costa09@usp.br,Sim,,,Marco Bregant,Sim,...,,03/11/2025 - 16:29,Feminino,Cisgênero (se identifica com o gênero associad...,Bissexual,Não,Branca,Médio completo,,Todo em escola pública (não técnica)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
169,victor paes plinio,Mestrado,14133,,victorpaes@usp.br,Sim,,,,,...,,07/11/2025 - 21:33,Masculino,Cisgênero (se identifica com o gênero associad...,Homossexual,Não,Branca,Fundamental incompleto,,Todo em escola pública (não técnica)
170,vinicius rodrigues jacinto dos santos,Mestrado,13792,,vinicius.rodrigues.santos@usp.br,Sim,,,,,...,,28/10/2025 - 15:27,Masculino,Cisgênero (se identifica com o gênero associad...,,Sim,Parda,Superior incompleto,,Todo em escola particular
171,vinicius zilio rocca,Doutorado,13987,,vinicius_rocca@hotmail.com,Sim,,,André Paniago Lessa,,...,Media não calculada devido conceitos incompatí...,06/11/2025 - 15:26,Masculino,Cisgênero (se identifica com o gênero associad...,Heterossexual,Não,Branca,Médio completo,,Todo em escola particular
172,willamis aragao da silva,Mestrado,14020,,willamis.10@hotmail.com,Sim,,,,,...,,06/11/2025 - 23:42,Masculino,Cisgênero (se identifica com o gênero associad...,Heterossexual,Não,Branca,Médio incompleto,,Parte em escola pública parte em particular


In [61]:
dfA = CPGPAR
dfA = dfA.rename(columns={'Nome':'Nome_'})

for key in dfs_euf[0:9]: # OBSERVAÇÂO: Normalmente seria [:9], mas houve 2 edições extras do EUF
  print(key, )
  dic[key]['Nome'] = dic[key]['Nome'].apply(processaString)
  dic[key]['Percentil'] = dic[key]['Percentil'].str.replace('%','').astype(float)
  dic[key]['Percentil'] = dic[key]['Percentil'].astype(float)   # remover posteriormente
  dic[key]['Nota'] = dic[key]['Nota'].astype(float)

  # Calcula a média da edição
  media_edicao = dic[key]['Nota'].mean()
  print(key,media_edicao)
  dic[key]['Nota_norm'] = dic[key]['Nota'].apply(lambda x: x*5/media_edicao)

2025_2
2025_2 4.64006456241033
2025_1
2025_1 3.8184496826835903
2024_2
2024_2 3.7599052540913007
2024_1
2024_1 3.609480269489894
2023_2
2023_2 3.5794205794205793
2023_1
2023_1 3.897113022113022
2022_2
2022_2 3.575174825174825
2022_1
2022_1 3.4540586245772267
2021_3
2021_3 4.0227272727272725


In [62]:
  dic['2024_2']['Nota_norm'] = dic['2024_2']['Nota'].apply(lambda x: x*5/3.74) # Utilizar essa linha, essa edição teve leve diferença

In [63]:
lista_colunas = ['Nome','Percentil','Nota','Nota_norm']# lista colunas de interesse
for key in dfs_euf[:9]:
  dfB = dic[key]
  # coluna_nome = key    # é necessário???
  # print(key)
  dfA = merge_RL(dfA,dfB,'Nome',lista_colunas,0.75) # 0.85 por padrão, depois rodar em menores.
  dfA = etiqueta_df(dfA, lista_colunas, key)


In [64]:
dfA

,Nome_,Curso,Nº de inscrição,Inst. origem,Email,Concorre à bolsa,EUF indicado,EUF antigo,Orientador Informado,Tem carta de aceite?,...,Nota_2022_2,Nota_norm_2022_2,Nome_2022_1,Percentil_2022_1,Nota_2022_1,Nota_norm_2022_1,Nome_2021_3,Percentil_2021_3,Nota_2021_3,Nota_norm_2021_3
0,alejandro lopez guilhermino,Mestrado,14058,,aleh.lopezg@gmail.com,Sim,,,Ana Julia Silveira Mizher,Sim,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,alex sander do carmo souza,Mestrado,13824,,alexsander6001@gmail.com,Sim,,,Elcio Abdalla,Nao,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,alexia oliveira silva,Doutorado,13847,,alexia.o@usp.br,Sim,,,Luiz Gustavo Pimenta Martins,Sim,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,alice affonso sassmannshausen,Mestrado,14023,,alicesass1009@hotmail.com,Sim,,,,,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,aline paulo da costa,Mestrado,13859,,aline_costa09@usp.br,Sim,,,Marco Bregant,Sim,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
181,victor paes plinio,Mestrado,14133,,victorpaes@usp.br,Sim,,,,,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
182,vinicius rodrigues jacinto dos santos,Mestrado,13792,,vinicius.rodrigues.santos@usp.br,Sim,,,,,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
183,vinicius zilio rocca,Doutorado,13987,,vinicius_rocca@hotmail.com,Sim,,,André Paniago Lessa,,...,NaN,NaN,vinicius zilio rocca,98.99,7.25,10.4949,NaN,NaN,NaN,NaN
184,willamis aragao da silva,Mestrado,14020,,willamis.10@hotmail.com,Sim,,,,,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [250]:
dfC = dfA.copy() # caso algo dê errado daqui para baixo, rode apenas esse celula

In [160]:
# Ordena as colunas
cabecalhos = list(dfC.columns)

cabecalhos_p1=['Nome_',
                'Curso',
                'Nº de inscrição',
                'Inst. origem',
                'Email',
                'Concorre à bolsa',
                'EUF indicado',
                'EUF antigo',
                'Orientador Informado',
                'Tem carta de aceite?',
                'Orientador Credenciado?',
                'Estará credenciado na data de matrícula?',
                'Observações',
                'Data de Envio',
                'Sexo Biológico',
                'Identidade de Gênero',
                'Orientação Sexual',
                'Pessoa com Deficiência (PcD*)?',
                'Cor ou raça*',
                'Nível educacional da mãe ou genitor primário',
                'Onde estudou a mãe (ou genitor primário) no ensino médio?',
                'Onde estudou no ensino médio?']

cabecalhos_p2 = ['Nota_maxima',
                'Nota_norm_maxima',
                'Percentil_maximo',
                'euf_selecionado',
                'Situação'
                ]

cabecalhos_p3 = [cab for cab in cabecalhos if cab not in cabecalhos_p1 + cabecalhos_p2]
cabecalhos_p3.sort()
cabecalhos_ordenados = cabecalhos_p1 + cabecalhos_p2 + cabecalhos_p3
# cabecalhos_p3

In [146]:
# não precisa rodar
cabecalhos_ordenados

['Nome_',
 'Curso',
 'Nº de inscrição',
 'Inst. origem',
 'Email',
 'Concorre à bolsa',
 'EUF indicado',
 'EUF antigo',
 'Orientador Informado',
 'Tem carta de aceite?',
 'Orientador Credenciado?',
 'Estará credenciado na data de matrícula?',
 'Observações',
 'Data de Envio',
 'Sexo Biológico',
 'Identidade de Gênero',
 'Orientação Sexual',
 'Pessoa com Deficiência (PcD*)?',
 'Cor ou raça*',
 'Nível educacional da mãe ou genitor primário',
 'Onde estudou a mãe (ou genitor primário) no ensino médio?',
 'Onde estudou no ensino médio?',
 'Nota_maxima',
 'Nota_norm_maxima',
 'Percentil_maximo',
 'euf_selecionado',
 'Situação',
 'Nome_2021_3',
 'Nome_2022_1',
 'Nome_2022_2',
 'Nome_2023_1',
 'Nome_2023_2',
 'Nome_2024_1',
 'Nome_2024_2',
 'Nome_2025_1',
 'Nome_2025_2',
 'Nota_2021_3',
 'Nota_2022_1',
 'Nota_2022_2',
 'Nota_2023_1',
 'Nota_2023_2',
 'Nota_2024_1',
 'Nota_2024_2',
 'Nota_2025_1',
 'Nota_2025_2',
 'Nota_norm_2021_3',
 'Nota_norm_2022_1',
 'Nota_norm_2022_2',
 'Nota_norm_2023_1

In [251]:
# Procura a maior nota
lista=list(dfC.columns)
string1 = 'Nota'
string2 = 'norm'
dfC['Nota_maxima'] = dfC[subtringLista(lista, string1, string2)].astype(float).max(axis=1)

string1 = 'Nota'
string2 = None
dfC['Nota_norm_maxima'] = dfC[subtringLista(lista, string1, string2)].astype(float).max(axis=1)

string1 = 'Percentil'
string2 = None
dfC['Percentil_maximo'] = dfC[subtringLista(lista, string1, string2)].astype(float).max(axis=1)

string1 = 'Percentil'

dfC['euf_selecionado'] = dfC[subtringLista(lista, string1, string2)].astype(float).idxmax(axis="columns")



# Verifica o curso e, se DD, aplica função alternativa (Not Norm Máx =>6)
dfC['Situação'] = dfC.apply(lambda linha: verificaNotaNorm(linha['Nota_norm_maxima']) if linha['Curso']!='Dou.Direto' else verificaNotaNormDD(linha['Nota_norm_maxima']) , axis=1)

dfC = dfC[cabecalhos_ordenados]

dfC = dfC.rename(columns = {'Nome_':'Nome',
                    'Pessoa com Deficiência (PcD*)?': 'Necessidades especiais',
                    'Nota_norm_maxima':'Nota norm. máx.',
                    'Cor ou raça*':'Cor ou raça',
                    'Nota_maxima': 'Nota máxima'
                    })



dfC['Nomes repetidos'] = dfC.groupby('Nome')['Nome'].transform('count')

nomes_duplicados = dfC[dfC['Nomes repetidos']>1]                        # Salva na variável a listas linhas repetidas

dfC = dfC.sort_values('Nº de inscrição').drop_duplicates(['Nome'], keep='last').reset_index()
dfC['Nome'] = dfC['Nome'].apply(capitalizarNome)

/tmp/ipykernel_1559/1266357091.py:17: FutureWarning: The behavior of DataFrame.idxmax with all-NA values, or any-NA and skipna=False, is deprecated. In a future version this will raise ValueError
  dfC['euf_selecionado'] = dfC[subtringLista(lista, string1, string2)].astype(float).idxmax(axis="columns")


In [252]:
dfC['Nomes repetidos'] = dfC.groupby('Nome')['Nome'].transform('count')
dfC['euf_selecionado'] = dfC['euf_selecionado'].apply(lambda x: str(x).replace('Percentil_',''))

In [253]:
dfC[dfC['Nomes repetidos']>1]  # Verifica se existem nomes repetidos

,index,Nome,Curso,Nº de inscrição,Inst. origem,Email,Concorre à bolsa,EUF indicado,EUF antigo,Orientador Informado,...,Percentil_2021_3,Percentil_2022_1,Percentil_2022_2,Percentil_2023_1,Percentil_2023_2,Percentil_2024_1,Percentil_2024_2,Percentil_2025_1,Percentil_2025_2,Nomes repetidos


In [254]:
dfC = dfC.sort_values('Nº de inscrição').drop_duplicates(['Nome'], keep='last').reset_index().drop(columns=['index','level_0'])

In [255]:
dfC.columns

Index(['Nome', 'Curso', 'Nº de inscrição', 'Inst. origem', 'Email',
       'Concorre à bolsa', 'EUF indicado', 'EUF antigo',
       'Orientador Informado', 'Tem carta de aceite?',
       'Orientador Credenciado?', 'Estará credenciado na data de matrícula?',
       'Observações', 'Data de Envio', 'Sexo Biológico',
       'Identidade de Gênero', 'Orientação Sexual', 'Necessidades especiais',
       'Cor ou raça', 'Nível educacional da mãe ou genitor primário',
       'Onde estudou a mãe (ou genitor primário) no ensino médio?',
       'Onde estudou no ensino médio?', 'Nota máxima', 'Nota norm. máx.',
       'Percentil_maximo', 'euf_selecionado', 'Situação', 'Nome_2021_3',
       'Nome_2022_1', 'Nome_2022_2', 'Nome_2023_1', 'Nome_2023_2',
       'Nome_2024_1', 'Nome_2024_2', 'Nome_2025_1', 'Nome_2025_2',
       'Nota_2021_3', 'Nota_2022_1', 'Nota_2022_2', 'Nota_2023_1',
       'Nota_2023_2', 'Nota_2024_1', 'Nota_2024_2', 'Nota_2025_1',
       'Nota_2025_2', 'Nota_norm_2021_3', 'Nota_norm_2

In [259]:
dfC['euf_selecionado'] = dfC['euf_selecionado'].replace({'nan' : 'EUF não encontrado'}) #EUF não encontrado
df_nao_encontrado = dfC[dfC['euf_selecionado'] == 'EUF não encontrado']
df_nao_encontrado = df_nao_encontrado[['Nome','Nota norm. máx.', 'euf_selecionado', 'Situação','Email']]
df_nao_encontrado.to_excel(f'INGRESSANTES_{edicao}_sem_EUF.xlsx',index=False)
df_nao_encontrado

,Nome,Nota norm. máx.,euf_selecionado,Situação,Email
9,Raza Muhammad,NaN,EUF não encontrado,Sem nota suficiente,razahi409@gmail.com
55,Suleiman Habibu Umar,NaN,EUF não encontrado,Sem nota suficiente,suleimanhabibuumar@gmail.com
104,Gary Wade Colley,NaN,EUF não encontrado,Sem nota suficiente,wade.colley@yale.edu
136,Mujahid Ali,NaN,EUF não encontrado,Sem nota suficiente,mujahid14.ali@gmail.com


In [249]:
# df_reprov = dfC[['Nome','Nota norm. máx.','Percentil_maximo', 'euf_selecionado', 'Situação','Email']]
# OU
#df_reprov = dfC
df_reprov = dfC[dfC['Situação']!='Aprovado']
df_reprov = df_reprov[['Nome','Nota norm. máx.','Percentil_maximo', 'euf_selecionado', 'Situação','Email']]
df_reprov.to_excel(f'INGRESSANTES_{edicao}_insuficiente.xlsx',index=False)
df_reprov # Apenas printar

,Nome,Nota norm. máx.,Percentil_maximo,euf_selecionado,Situação,Email
8,Amanda Mestico Lacerda,1.885750,1.00,2025_2,Sem nota suficiente,amandamestico@gmail.comn
9,Raza Muhammad,NaN,NaN,EUF não encontrado,Sem nota suficiente,razahi409@gmail.com
23,Davinny Semilly Gama Neto Da Silva,3.232714,17.72,2025_2,Sem nota suficiente,davinnysgama@gmail.com
55,Suleiman Habibu Umar,NaN,NaN,EUF não encontrado,Sem nota suficiente,suleimanhabibuumar@gmail.com
104,Gary Wade Colley,NaN,NaN,EUF não encontrado,Sem nota suficiente,wade.colley@yale.edu
136,Mujahid Ali,NaN,NaN,EUF não encontrado,Sem nota suficiente,mujahid14.ali@gmail.com
137,Erick Syuffi Lagedo,3.502106,23.96,2025_2,Sem nota suficiente,ericksyuffi@hotmail.com
152,Thaina Calaca Futamata,3.928296,38.71,2025_1,Sem nota suficiente,thainafutamata@usp.br


In [75]:
# Gera o excel
df_aprov = dfC[['Nº de inscrição','Nome','Curso','Nota norm. máx.','Percentil_maximo','euf_selecionado','Concorre à bolsa', 'Situação','Email']]
df_aprov = df_aprov[df_aprov['Situação']=='Aprovado']
df_aprov.to_excel(f'INGRESSANTES - Aprovados - {edicao}.xlsx',index=False)

In [76]:
df_euf_selecionado = df_aprov[df_aprov['Concorre à bolsa']=='Sim']

In [77]:
# Gera o excel
df_euf_selecionado[['Nº de inscrição','Curso','euf_selecionado']].to_excel(f'INGRESSANTES - Lista de EUF selecionado - {edicao}.xlsx',index=False)
df_euf_selecionado

,Nº de inscrição,Nome,Curso,Nota norm. máx.,Percentil_maximo,euf_selecionado,Concorre à bolsa,Situação,Email
0,13748,Hamed Moayyedi,Doutorado,5.892444,72.53,2025_2,Sim,Aprovado,hamedmoayyedi@gmail.com
1,13750,Marcos Paulo Attanasio Filho,Mestrado,9.493382,96.83,2024_2,Sim,Aprovado,m.attanasio@usp.br
2,13753,Juliana Duarte Da Silva Correia,Mestrado,4.040892,38.71,2024_2,Sim,Aprovado,julli.duarte96@gmail.com
3,13754,Ederson Neri De Araujo,Mestrado,4.190622,43.76,2024_2,Sim,Aprovado,edersonnrj@gmail.com
4,13757,Gustavo Fregolente Micossi,Mestrado,5.926642,74.89,2025_1,Sim,Aprovado,gustavof.micossi@gmail.com
...,...,...,...,...,...,...,...,...,...
164,14148,Fernando Oliveira Credito,Mestrado,6.734820,85.08,2025_1,Sim,Aprovado,fernando9.credito@usp.br
165,14149,Thiago Hideki Tsutiya Guenka,Mestrado,5.657249,71.16,2025_1,Sim,Aprovado,thiago.tsutiya@aluno.ufabc.edu.br
166,14150,Benjamin Giriboni Monteiro,Mestrado,7.273606,89.02,2024_2,Sim,Aprovado,benjamin.giriboni.monteiro@gmail.com
167,14151,Gabriel Quaglio Morales Sanchez,Mestrado,9.159355,97.99,2025_1,Sim,Aprovado,gabrielsanchez@usp.br


In [78]:
# Inserir "Não respondeu" em celulas em branco do questionário socio econômico

colunas = ['Sexo Biológico',
           'Identidade de Gênero', 'Orientação Sexual', 'Necessidades especiais',
           'Cor ou raça', 'Nível educacional da mãe ou genitor primário',
           'Onde estudou a mãe (ou genitor primário) no ensino médio?',
           'Onde estudou no ensino médio?']
# dfC.columns
for coluna in colunas:
  dfC[coluna].replace('',"Não respondeu", inplace=True)

/tmp/ipykernel_1559/2848319095.py:10: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  dfC[coluna].replace('',"Não respondeu", inplace=True)


In [79]:
# Gera o excel

dfC.to_excel(f'INGRESSANTES - EUFs e inscrições do CPGpar - {edicao}.xlsx', index=False)